<a href="https://colab.research.google.com/github/MsTejasviSingh/Cancer-Genomics/blob/main/01_RNAseq_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNA-Seq Data Analysis Project

## Project Goal

The goal of this project is to learn and practice a complete RNA-sequencing (RNA-seq) bioinformatics workflow using publicly available sequencing data.

This project will cover:

- Raw FASTQ file quality control (FastQC)
- Read quality assessment
- Adapter and quality trimming
- Sequence alignment
- Gene expression quantification
- Exploratory Data Analysis (EDA)
- Data visualization
- Statistical analysis
- Machine Learning applications

The project is performed in Google Colab and all data are stored in Google Drive for persistence across sessions.

# Dataset Download

The dataset used in this project is the RNA-seq example dataset provided by Johns Hopkins University.

Source:
ftp://ftp.ccb.jhu.edu/pub/RNAseq_protocol/chrX_data.tar.gz

The dataset contains:

- Raw RNA-seq FASTQ files
- Human chromosome X reference genome
- Gene annotation files
- Sample metadata
- HISAT2 alignment indexes

The compressed archive size is approximately 2 GB.

In [ ]:
!wget ftp://ftp.ccb.jhu.edu/pub/RNAseq_protocol/chrX_data.tar.gz

--2026-08-20 16:26:07--  ftp://ftp.ccb.jhu.edu/pub/RNAseq_protocol/chrX_data.tar.gz
           => ‘chrX_data.tar.gz’
Resolving ftp.ccb.jhu.edu (ftp.ccb.jhu.edu)... 128.220.174.63
Connecting to ftp.ccb.jhu.edu (ftp.ccb.jhu.edu)|128.220.174.63|:21... connected.
Logging in as anonymous ... Logged in!
==> SYST ... done.    ==> PWD ... done.
==> TYPE I ... done.  ==> CWD (1) /pub/RNAseq_protocol ... done.
==> SIZE chrX_data.tar.gz ... 2113551170
==> PASV ... done.    ==> RETR chrX_data.tar.gz ... done.
Length: 2113551170 (2.0G) (unauthoritative)

chrX_data.tar.gz    100%[===================>]   1.97G  3.72MB/s    in 14m 3s  

^C


# Dataset Extraction

The downloaded archive is extracted using the Linux `tar` command.

After extraction, a directory called `chrX_data` is created containing all files required for RNA-seq analysis.

In [ ]:
!tar xvzf chrX_data.tar.gz

chrX_data/
chrX_data/genes/
chrX_data/genes/chrX.gtf
chrX_data/genome/
chrX_data/genome/chrX.fa
chrX_data/indexes/
chrX_data/indexes/chrX_tran.3.ht2
chrX_data/indexes/chrX_tran.4.ht2
chrX_data/indexes/chrX_tran.1.ht2
chrX_data/indexes/chrX_tran.2.ht2
chrX_data/indexes/chrX_tran.7.ht2
chrX_data/indexes/chrX_tran.8.ht2
chrX_data/indexes/chrX_tran.5.ht2
chrX_data/indexes/chrX_tran.6.ht2
chrX_data/samples/
chrX_data/samples/ERR188044_chrX_1.fastq.gz
chrX_data/samples/ERR188044_chrX_2.fastq.gz
chrX_data/samples/ERR188104_chrX_1.fastq.gz
chrX_data/samples/ERR188104_chrX_2.fastq.gz
chrX_data/samples/ERR188234_chrX_1.fastq.gz
chrX_data/samples/ERR188234_chrX_2.fastq.gz
chrX_data/samples/ERR188245_chrX_1.fastq.gz
chrX_data/samples/ERR188245_chrX_2.fastq.gz
chrX_data/samples/ERR188257_chrX_1.fastq.gz
chrX_data/samples/ERR188257_chrX_2.fastq.gz
chrX_data/samples/ERR188273_chrX_1.fastq.gz
chrX_data/samples/ERR188273_chrX_2.fastq.gz
chrX_data/samples/ERR188337_chrX_1.fastq.gz
chrX_data/samples/ERR1

# Dataset Structure Inspection

After extraction, the contents of the dataset are inspected to verify successful download and extraction.

Linux commands are used to examine:

- Folder structure
- File sizes
- Dataset organization

In [ ]:
!ls -lh

total 2.0G
drwxrwxr-x 6 2612 5032 4.0K Jan 14  2016 chrX_data
-rw-r--r-- 1 root root 2.0G Aug 20 16:40 chrX_data.tar.gz
drwx------ 5 root root 4.0K Aug 20 16:25 drive
drwxr-xr-x 1 root root 4.0K Aug 18 13:44 sample_data


In [ ]:
!ls -lh chrX_data

total 24K
drwxrwxr-x 2 2612 5032 4.0K Jul 12  2016 genes
drwxrwxr-x 2 2612 5032 4.0K Jan 14  2016 genome
-rw-r--r-- 1 2612 5032  337 Jan 14  2016 geuvadis_phenodata.csv
drwxrwxr-x 2 2612 5032 4.0K Jan 14  2016 indexes
-rw-rw-r-- 1 2612 5032  228 Jan 14  2016 mergelist.txt
drwxrwxr-x 2 2612 5032 4.0K Jan 14  2016 samples


# Dataset Size Analysis

The size of each component of the dataset is evaluated.

This helps identify:

- Storage requirements
- Largest files
- Location of raw sequencing data

The majority of the dataset size is expected to come from the FASTQ files stored in the `samples` directory.

In [ ]:
!du -sh chrX_data/*

6.5M	chrX_data/genes
152M	chrX_data/genome
4.0K	chrX_data/geuvadis_phenodata.csv
272M	chrX_data/indexes
4.0K	chrX_data/mergelist.txt
1.8G	chrX_data/samples


In [ ]:
!ls -lh chrX_data/samples

total 1.8G
-rw-rw-r-- 1 2612 5032  88M Jan 14  2016 ERR188044_chrX_1.fastq.gz
-rw-rw-r-- 1 2612 5032  88M Jan 14  2016 ERR188044_chrX_2.fastq.gz
-rw-rw-r-- 1 2612 5032  84M Jan 14  2016 ERR188104_chrX_1.fastq.gz
-rw-rw-r-- 1 2612 5032  85M Jan 14  2016 ERR188104_chrX_2.fastq.gz
-rw-rw-r-- 1 2612 5032 108M Jan 14  2016 ERR188234_chrX_1.fastq.gz
-rw-rw-r-- 1 2612 5032 109M Jan 14  2016 ERR188234_chrX_2.fastq.gz
-rw-rw-r-- 1 2612 5032  59M Jan 14  2016 ERR188245_chrX_1.fastq.gz
-rw-rw-r-- 1 2612 5032  60M Jan 14  2016 ERR188245_chrX_2.fastq.gz
-rw-rw-r-- 1 2612 5032  65M Jan 14  2016 ERR188257_chrX_1.fastq.gz
-rw-rw-r-- 1 2612 5032  66M Jan 14  2016 ERR188257_chrX_2.fastq.gz
-rw-rw-r-- 1 2612 5032  39M Jan 14  2016 ERR188273_chrX_1.fastq.gz
-rw-rw-r-- 1 2612 5032  39M Jan 14  2016 ERR188273_chrX_2.fastq.gz
-rw-rw-r-- 1 2612 5032  88M Jan 14  2016 ERR188337_chrX_1.fastq.gz
-rw-rw-r-- 1 2612 5032  88M Jan 14  2016 ERR188337_chrX_2.fastq.gz
-rw-rw-r-- 1 2612 5032  63M Jan 14  2016 ERR188383_

In [ ]:
!ls /content/drive/MyDrive

'10. PMP Exam Prep_Risk.pdf'
'4 country tour.gsheet'
'Airbnb NYC 2019.csv'
'Application Payment (1).gdoc'
'Application Payment (2).gdoc'
'Application Payment.gdoc'
 bioinformatics-tool-ecosystem-mastery-path.md.gdoc
'Colab Notebooks'
 genomics-file-formats-mastery-path.md.gdoc
 German
'How to get started with Drive.pdf'
'Mobile Number Update - English User guide - expat.gdoc'
 Shhh
 Tejasvi_Singh_Resume_ALS.gdoc
 Tejasvi_Singh_Resume_Flychem_RDChemist.docx
 Tejasvi_Singh_Resume_Physical_TwoVersions.gdoc
'Untitled document.gdoc'


# Google Drive Storage

Google Colab sessions are temporary and all local files are deleted when the session ends.

To prevent repeated downloading of the 2 GB dataset, the entire dataset is copied to Google Drive.

This allows future analysis sessions to directly access the data without repeating the download step.

In [ ]:
!mkdir -p "/content/drive/MyDrive/RNAseq_Project"

In [ ]:
!cp -r chrX_data "/content/drive/MyDrive/RNAseq_Project"

In [ ]:
!du -sh "/content/drive/MyDrive/RNAseq_Project/chrX_data"

2.2G	/content/drive/MyDrive/RNAseq_Project/chrX_data


## Why Perform Quality Control?

Raw sequencing data can contain:

- Low-quality bases
- Sequencing errors
- Adapter contamination
- PCR duplicates
- Technical artifacts

Before performing alignment, gene expression analysis, or machine learning, the quality of the sequencing reads must be assessed.

FastQC provides a visual summary of sequencing quality and helps identify potential problems in the dataset.

In this project, FastQC will be used to evaluate the quality of the raw RNA-seq reads before any downstream analysis.

# FastQC Quality Control

FastQC is a widely used bioinformatics quality control tool for sequencing data.

The purpose of FastQC is to evaluate the quality of raw sequencing reads before any downstream analysis.

Key metrics evaluated include:

- Per-base sequence quality
- Per-sequence quality scores
- GC content distribution
- Sequence duplication levels
- Adapter contamination
- Read length distribution

Quality assessment is an essential first step in any RNA-seq workflow because poor-quality reads can negatively affect alignment, quantification, and downstream biological interpretation.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
data_path= "/content/drive/MyDrive/RNAseq_Project/chrX_data"

## FastQC Output Directory

A dedicated output directory is created to store FastQC reports.

Keeping quality-control outputs separate from raw sequencing data improves project organization and reproducibility.

All FastQC reports generated during this project will be stored in:

`RNAseq_Project/FastQC_results`

In [ ]:
!mkdir -p "/content/drive/MyDrive/RNAseq_Project/FastQC_results"

In [ ]:
results_path = "/content/drive/MyDrive/RNAseq_Project/FastQC_results"

In [ ]:
print("Data folder:", data_path)
print("Results folder:", results_path)

Data folder: /content/drive/MyDrive/RNAseq_Project/chrX_data
Results folder: /content/drive/MyDrive/RNAseq_Project/FastQC_results


## FastQC Installation

Google Colab provides a temporary computing environment.

Software installed during a previous session is not retained after the session ends.

Therefore, FastQC must be installed at the beginning of each new Colab session before performing quality-control analysis.

In [ ]:
!apt-get -qq update
!apt-get -qq install fastqc

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libatspi2.0-0:amd64.
(Reading database ... 118422 files and directories currently installed.)
Preparing to unpack .../00-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../01-libxtst6_2%3a1.2.3-1build4_amd64.deb ...
Unpacking libxtst6:amd64 (2:1.2.3-1build4) ...
Selecting previously unselected package session-migration.
Preparing to unpack .../02-session-migration_0.3.6_amd64.deb ...
Unpacking session-migration (0.3.6) ...
Selecting previously unselected package gsettings-desktop-schemas.
Preparing to unpack .../03-gsettings-desktop-schemas_42.0-1ubuntu1_all.deb ...
Unpacking gsettings-desktop-schemas (42.0-1ubu

In [ ]:
!fastqc --version

[0.025s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.025s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
FastQC v0.11.9


## Running FastQC on a Raw RNA-seq Sample

FastQC is performed on a single FASTQ file to evaluate the quality of the raw sequencing reads.

The selected file:

`ERR188044_chrX_1.fastq.gz`

contains the forward reads (Read 1) of one paired-end RNA-seq sample.

The FastQC report will provide information about:

- Per-base sequence quality
- Per-sequence quality scores
- GC content
- Sequence duplication levels
- Adapter contamination
- Read length distribution

The generated reports will be saved in the `FastQC_results` directory for further inspection.

In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project"

total 12K
drwx------ 2 root root 4.0K Aug 20 16:55 chrX_data
drwx------ 2 root root 4.0K Aug 21 16:41 FastQC_results
drwx------ 2 root root 4.0K Aug 21 17:53 MultiQC


In [ ]:
!fastqc \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188044_chrX_1.fastq.gz" \ -o "/content/drive/MyDrive/RNAseq_Project/FastQC_results/"

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Skipping ' -o' which didn't exist, or couldn't be read
Started analysis of ERR188044_chrX_1.fastq.gz
Failed to process /content/drive/MyDrive/RNAseq_Project/FastQC_results
java.io.FileNotFoundException: /content/drive/MyDrive/RNAseq_Project/FastQC_results (Is a directory)
	at java.base/java.io.FileInputStream.open0(Native Method)
	at java.base/java.io.FileInputStream.open(FileInputStream.java:213)
	at java.base/java.io.FileInputStream.<init>(FileInputStream.java:152)
	at uk.ac.babraham.FastQC.Sequence.FastQFile.<init>(FastQFile.java:73)
	at uk.ac.babraham.FastQC.Sequence.SequenceFactory.getSequenceFile(SequenceFactory.java:106)
	at uk.ac.babraham.FastQC.Sequence.

In [ ]:
!mv "/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188044_chrX_1_fastqc.html" "/content/drive/MyDrive/RNAseq_Project/FastQC_results/"

!mv "/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188044_chrX_1_fastqc.zip" "/content/drive/MyDrive/RNAseq_Project/FastQC_results/"

In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/FastQC_results"

total 2.1M
-rw------- 1 root root 637K Aug 21 17:08 ERR188044_chrX_1_fastqc.html
-rw------- 1 root root 422K Aug 21 17:08 ERR188044_chrX_1_fastqc.zip
-rw------- 1 root root 636K Aug 21 17:18 ERR188044_chrX_2_fastqc.html
-rw------- 1 root root 418K Aug 21 17:18 ERR188044_chrX_2_fastqc.zip


In [ ]:
!fastqc -o "/content/drive/MyDrive/RNAseq_Project/FastQC_results" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188104_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188104_chrX_2.fastq.gz"

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Started analysis of ERR188104_chrX_1.fastq.gz
Approx 5% complete for ERR188104_chrX_1.fastq.gz
Approx 10% complete for ERR188104_chrX_1.fastq.gz
Approx 15% complete for ERR188104_chrX_1.fastq.gz
Approx 20% complete for ERR188104_chrX_1.fastq.gz
Approx 25% complete for ERR188104_chrX_1.fastq.gz
Approx 30% complete for ERR188104_chrX_1.fastq.gz
Approx 35% complete for ERR188104_chrX_1.fastq.gz
Approx 40% complete for ERR188104_chrX_1.fastq.gz
Approx 45% complete for ERR188104_chrX_1.fastq.gz
Approx 50% complete for ERR188104_chrX_1.fastq.gz
Approx 55% complete for ERR188104_chrX_1.fastq.gz
Approx 60% complete for ERR188104_chrX_1.fastq.gz
Approx 65% complete for ER

In [ ]:
!fastqc -o "/content/drive/MyDrive/RNAseq_Project/FastQC_results" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188234_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188234_chrX_2.fastq.gz"

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Started analysis of ERR188234_chrX_1.fastq.gz
Approx 5% complete for ERR188234_chrX_1.fastq.gz
Approx 10% complete for ERR188234_chrX_1.fastq.gz
Approx 15% complete for ERR188234_chrX_1.fastq.gz
Approx 20% complete for ERR188234_chrX_1.fastq.gz
Approx 25% complete for ERR188234_chrX_1.fastq.gz
Approx 30% complete for ERR188234_chrX_1.fastq.gz
Approx 35% complete for ERR188234_chrX_1.fastq.gz
Approx 40% complete for ERR188234_chrX_1.fastq.gz
Approx 45% complete for ERR188234_chrX_1.fastq.gz
Approx 50% complete for ERR188234_chrX_1.fastq.gz
Approx 55% complete for ERR188234_chrX_1.fastq.gz
Approx 60% complete for ERR188234_chrX_1.fastq.gz
Approx 65% complete for ER

In [ ]:
!ls "/content/drive/MyDrive/RNAseq_Project/FastQC_results" | wc -l

12


In [ ]:
!fastqc -o "/content/drive/MyDrive/RNAseq_Project/FastQC_results" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188245_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188245_chrX_2.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188257_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188257_chrX_2.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188273_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188273_chrX_2.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188337_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188337_chrX_2.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188383_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188383_chrX_2.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188401_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188401_chrX_2.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188428_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188428_chrX_2.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188454_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188454_chrX_2.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR204916_chrX_1.fastq.gz" \
"/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR204916_chrX_2.fastq.gz"

[0.002s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.002s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Started analysis of ERR188245_chrX_1.fastq.gz
Approx 5% complete for ERR188245_chrX_1.fastq.gz
Approx 10% complete for ERR188245_chrX_1.fastq.gz
Approx 15% complete for ERR188245_chrX_1.fastq.gz
Approx 20% complete for ERR188245_chrX_1.fastq.gz
Approx 25% complete for ERR188245_chrX_1.fastq.gz
Approx 30% complete for ERR188245_chrX_1.fastq.gz
Approx 35% complete for ERR188245_chrX_1.fastq.gz
Approx 40% complete for ERR188245_chrX_1.fastq.gz
Approx 45% complete for ERR188245_chrX_1.fastq.gz
Approx 50% complete for ERR188245_chrX_1.fastq.gz
Approx 55% complete for ERR188245_chrX_1.fastq.gz
Approx 60% complete for ERR188245_chrX_1.fastq.gz
Approx 65% complete for ER

In [ ]:
!ls "/content/drive/MyDrive/RNAseq_Project/FastQC_results" | wc -l

48


## Batch FastQC Analysis Completed

FastQC quality-control analysis was successfully performed on all RNA-seq FASTQ files in the dataset.

Dataset summary:

- Total RNA-seq files analyzed: 24
- Total FastQC HTML reports generated: 24
- Total FastQC ZIP reports generated: 24
- Total FastQC output files: 48

All reports were stored in the `FastQC_results` directory and will be summarized in a subsequent MultiQC analysis.

## MultiQC Analysis

FastQC generates an individual report for each sequencing file. Since this dataset contains multiple samples, reviewing each report separately can be time-consuming.

MultiQC aggregates all FastQC reports into a single interactive report, allowing rapid assessment of sequencing quality across the entire dataset.

In [ ]:
!pip install multiqc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 66.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 MB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 6.4 MB/s eta 0:00:00


In [ ]:
!multiqc --version

multiqc, version 1.35


In [ ]:
!ls "/content/drive/MyDrive/RNAseq_Project/FastQC_results" | head

ERR188044_chrX_1_fastqc.html
ERR188044_chrX_1_fastqc.zip
ERR188044_chrX_2_fastqc.html
ERR188044_chrX_2_fastqc.zip
ERR188104_chrX_1_fastqc.html
ERR188104_chrX_1_fastqc.zip
ERR188104_chrX_2_fastqc.html
ERR188104_chrX_2_fastqc.zip
ERR188234_chrX_1_fastqc.html
ERR188234_chrX_1_fastqc.zip


In [ ]:
!mkdir -p "/content/drive/MyDrive/RNAseq_Project/MultiQC"

In [ ]:
!which multiqc

/usr/local/bin/multiqc


In [ ]:
!multiqc --help | head

                                                                                
 /// MultiQC 🔍 v1.35                                                           
                                                                                
 Usage: multiqc [OPTIONS] [ANALYSIS DIRECTORY]                                  
                                                                                
 MultiQC aggregates results from bioinformatics analyses across many samples    
 into a single report.                                                          
 It searches a given directory for analysis logs and compiles an HTML report.   
 It's a general use tool, perfect for summarising the output from numerous      
 bioinformatics tools.                                                          
Exception ignored on flushing sys.stdout:
BrokenPipeError: [Errno 32] Broken pipe


In [ ]:
!multiqc "/content/drive/MyDrive/RNAseq_Project/FastQC_results" \ -o "/content/drive/MyDrive/RNAseq_Project/MultiQC"

                                                                                
 /// ]8;id=593558;https://multiqc.info\MultiQC]8;;\ 🔍 v1.35                                                           
                                                                                
 Usage: multiqc [OPTIONS] [ANALYSIS DIRECTORY]                                  
                                                                                
 This is MultiQC v1.35                                                          
 For more help, run 'multiqc --help' or visit ]8;id=919967;http://multiqc.info\http://multiqc.info]8;;\               
╭─ Error ──────────────────────────────────────────────────────────────────────╮
│ Invalid value for '[ANALYSIS DIRECTORY]': Path ' -o' does not exist.         │
╰──────────────────────────────────────────────────────────────────────────────╯
                                                                                


In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/MultiQC"

total 2.7M
drwx------ 2 root root 4.0K Aug 21 17:55 multiqc_data
-rw------- 1 root root 2.7M Aug 21 17:55 multiqc_report.html


In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/MultiQC"

total 2.7M
drwx------ 2 root root 4.0K Aug 21 17:55 multiqc_data
-rw------- 1 root root 2.7M Aug 21 17:55 multiqc_report.html


In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/MultiQC"

total 2.7M
drwx------ 2 root root 4.0K Aug 21 17:55 multiqc_data
-rw------- 1 root root 2.7M Aug 21 17:55 multiqc_report.html


## MultiQC Results

MultiQC successfully aggregated all FastQC reports into a single summary report.

Generated files:

- `multiqc_report.html` : Interactive HTML report containing quality metrics across all samples.
- `multiqc_data/` : Directory containing the summarized quality control statistics used to generate the report.

This report provides a dataset-wide overview of sequencing quality and helps identify any systematic issues before downstream analysis.

# Read Alignment with HISAT2

After quality control, sequencing reads are aligned to a reference transcriptome.

Alignment maps each sequencing read to its most likely origin in the reference sequence.

The provided dataset contains pre-built HISAT2 index files (`chrX_tran.*.ht2`), allowing direct alignment without the computationally expensive index-building step.

Output:
- SAM (Sequence Alignment Map) file containing aligned reads.


In [ ]:
!apt-get install -y hisat2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  bcftools libhts3 libhtscodecs2 python3-hisat2 samtools
Suggested packages:
  python3-numpy python3-matplotlib texlive-latex-recommended cwltool
The following NEW packages will be installed:
  bcftools hisat2 libhts3 libhtscodecs2 python3-hisat2 samtools
0 upgraded, 6 newly installed, 0 to remove and 4 not upgraded.
Need to get 5,505 kB of archives.
After this operation, 17.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhtscodecs2 amd64 1.1.1-3 [53.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhts3 amd64 1.13+ds-2build1 [390 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 bcftools amd64 1.13-1 [697 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 hisat2 amd64 2.2.1-3 [3,832 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/uni

In [ ]:
!hisat2 --version

/usr/bin/hisat2-align-s version 2.2.1
64-bit
Built on Debian
28 September 2021
Compiler: gcc version 11.2.0 (Ubuntu 11.2.0-7ubuntu2) 
Options: -O3   -funroll-loops -g3 -Wdate-time -D_FORTIFY_SOURCE=2 -std=c++11
Sizeof {int, long, long long, void*, size_t, off_t}: {4, 8, 8, 8, 8, 8}


In [ ]:
!mkdir -p "/content/drive/MyDrive/RNAseq_Project/Alignment_results"

In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project"

total 16K
drwx------ 2 root root 4.0K Aug 22 08:10 Alignment_results
drwx------ 2 root root 4.0K Aug 20 16:55 chrX_data
drwx------ 2 root root 4.0K Aug 21 16:41 FastQC_results
drwx------ 3 root root 4.0K Aug 21 17:53 MultiQC


In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/chrX_data/indexes"

total 272M
-rw------- 1 root root  95M Aug 20 16:55 chrX_tran.1.ht2
-rw------- 1 root root  37M Aug 20 16:55 chrX_tran.2.ht2
-rw------- 1 root root  314 Aug 20 16:55 chrX_tran.3.ht2
-rw------- 1 root root  37M Aug 20 16:55 chrX_tran.4.ht2
-rw------- 1 root root  66M Aug 20 16:55 chrX_tran.5.ht2
-rw------- 1 root root  38M Aug 20 16:55 chrX_tran.6.ht2
-rw------- 1 root root  44K Aug 20 16:55 chrX_tran.7.ht2
-rw------- 1 root root 9.0K Aug 20 16:55 chrX_tran.8.ht2


In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/chrX_data/samples" | head

total 1.8G
-rw------- 1 root root  88M Aug 20 16:55 ERR188044_chrX_1.fastq.gz
-rw------- 1 root root  88M Aug 20 16:55 ERR188044_chrX_2.fastq.gz
-rw------- 1 root root  84M Aug 20 16:55 ERR188104_chrX_1.fastq.gz
-rw------- 1 root root  85M Aug 20 16:55 ERR188104_chrX_2.fastq.gz
-rw------- 1 root root 108M Aug 20 16:55 ERR188234_chrX_1.fastq.gz
-rw------- 1 root root 109M Aug 20 16:55 ERR188234_chrX_2.fastq.gz
-rw------- 1 root root  59M Aug 20 16:55 ERR188245_chrX_1.fastq.gz
-rw------- 1 root root  60M Aug 20 16:55 ERR188245_chrX_2.fastq.gz
-rw------- 1 root root  65M Aug 20 16:55 ERR188257_chrX_1.fastq.gz


## Local Alignment Output

The alignment output is initially written to Colab's local storage (`/content`) rather than directly to Google Drive.

This reduces input/output overhead and can significantly improve performance when generating large SAM files.

Once alignment is complete, the resulting file can be copied to Google Drive for permanent storage.

In [ ]:
!mkdir -p /content/alignment_test

In [ ]:
!hisat2 -p 2 \
-x "/content/drive/MyDrive/RNAseq_Project/chrX_data/indexes/chrX_tran" \
-1 "/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188044_chrX_1.fastq.gz" \
-2 "/content/drive/MyDrive/RNAseq_Project/chrX_data/samples/ERR188044_chrX_2.fastq.gz" \
-S "/content/alignment_test/ERR188044.sam"

1321477 reads; of these:
  1321477 (100.00%) were paired; of these:
    112729 (8.53%) aligned concordantly 0 times
    1185061 (89.68%) aligned concordantly exactly 1 time
    23687 (1.79%) aligned concordantly >1 times
    ----
    112729 pairs aligned concordantly 0 times; of these:
      4530 (4.02%) aligned discordantly 1 time
    ----
    108199 pairs aligned 0 times concordantly or discordantly; of these:
      216398 mates make up the pairs; of these:
        109001 (50.37%) aligned 0 times
        104638 (48.35%) aligned exactly 1 time
        2759 (1.27%) aligned >1 times
95.88% overall alignment rate


In [ ]:
!cp /content/alignment_test/ERR188044.sam \
"/content/drive/MyDrive/RNAseq_Project/Alignment_results"

In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/Alignment_results"

total 747M
-rw------- 1 root root 747M Aug 22 14:53 ERR188044.sam


# Converting SAM to BAM

The SAM alignment file was converted to BAM format using samtools.

BAM (Binary Alignment Map) is a compressed binary representation of SAM files that reduces storage requirements and improves computational efficiency for downstream analyses.

Input:
- ERR188044.sam

Output:
- ERR188044.bam

In [ ]:
!apt-get install -y samtools

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
samtools is already the newest version (1.13-4).
samtools set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 4 not upgraded.


In [ ]:
!samtools --version

samtools 1.13
Using htslib 1.13+ds
Copyright (C) 2021 Genome Research Ltd.

Samtools compilation details:
    Features:       build=configure curses=yes 
    CC:             gcc
    CPPFLAGS:       -frelease  -Wdate-time -D_FORTIFY_SOURCE=2
    CFLAGS:         -g -O2 -ffile-prefix-map=�BUILDPATH�=. -flto=auto -ffat-lto-objects -fstack-protector-strong -Wformat -Werror=format-security
    LDFLAGS:        -Wl,-Bsymbolic-functions -flto=auto -Wl,-z,relro -Wl,-z,now
    HTSDIR:         
    LIBS:           
    CURSES_LIB:     -lcurses

HTSlib compilation details:
    Features:       build=configure plugins=yes, plugin-path=/usr/local/lib/htslib:/usr/local/libexec/htslib:: libcurl=yes S3=yes GCS=yes libdeflate=yes lzma=yes bzip2=yes htscodecs=1.1.1
    CC:             gcc
    CPPFLAGS:       -I. -DSAMTOOLS=1 -Wdate-time -D_FORTIFY_SOURCE=2
    CFLAGS:         -g -O2 -ffile-prefix-map=/build/htslib-TQtOKr/htslib-1.13+ds=. -flto=auto -ffat-lto-objects -fstack-protector-strong -Wformat -Werro

In [ ]:
!samtools view -b \
/content/alignment_test/ERR188044.sam \
> /content/alignment_test/ERR188044.bam

In [ ]:
!ls -lh /content/alignment_test/

total 957M
-rw-r--r-- 1 root root 211M Aug 22 15:04 ERR188044.bam
-rw-r--r-- 1 root root 747M Aug 22 14:46 ERR188044.sam


# SAM to BAM Conversion

The alignment output generated by HISAT2 was initially stored in SAM format. Since SAM files are large and inefficient for downstream analysis, the file was converted to BAM format using samtools.

### Observation

The BAM file occupies substantially less storage space while retaining the same alignment information. BAM is therefore the preferred format for downstream RNA-seq analysis.

In [ ]:
!mkdir -p "/content/drive/MyDrive/RNAseq_Project/BAM_results"

In [ ]:
!cp /content/alignment_test/ERR188044.bam "/content/drive/MyDrive/RNAseq_Project/BAM_results"

In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/BAM_results"

total 211M
-rw------- 1 root root 211M Aug 22 15:08 ERR188044.bam


In [ ]:
%%bash
mkdir -p /content/alignment_test/


for r1 in /content/drive/MyDrive/RNAseq_Project/chrX_data/samples/*_1.fastq.gz
do
r2=${r1/_1.fastq.gz/_2.fastq.gz}


sample=$(basename "$r1" _1.fastq.gz)

echo "Processing $sample"

hisat2 -p 2 \
-x "/content/drive/MyDrive/RNAseq_Project/chrX_data/indexes/chrX_tran" \
-1 "$r1" \
-2 "$r2" \
-S "/content/alignment_test/${sample}.sam"

samtools view -b \
"/content/alignment_test/${sample}.sam" \
>"/content/alignment_test/${sample}.bam"


rm "/content/alignment_test/${sample}.sam"


cp "/content/alignment_test/${sample}.bam" \ "content/drive/MyDrive/RNAseq_Project/BAM_results/"

done

Processing ERR188044_chrX
Processing ERR188104_chrX
Processing ERR188234_chrX
Processing ERR188245_chrX
Processing ERR188257_chrX
Processing ERR188273_chrX
Processing ERR188337_chrX
Processing ERR188383_chrX
Processing ERR188401_chrX
Processing ERR188428_chrX
Processing ERR188454_chrX
Processing ERR204916_chrX


1321477 reads; of these:
  1321477 (100.00%) were paired; of these:
    112729 (8.53%) aligned concordantly 0 times
    1185061 (89.68%) aligned concordantly exactly 1 time
    23687 (1.79%) aligned concordantly >1 times
    ----
    112729 pairs aligned concordantly 0 times; of these:
      4530 (4.02%) aligned discordantly 1 time
    ----
    108199 pairs aligned 0 times concordantly or discordantly; of these:
      216398 mates make up the pairs; of these:
        109001 (50.37%) aligned 0 times
        104638 (48.35%) aligned exactly 1 time
        2759 (1.27%) aligned >1 times
95.88% overall alignment rate
cp: cannot create regular file ' content/drive/MyDrive/RNAseq_Project/BAM_results/': No such file or directory
1292343 reads; of these:
  1292343 (100.00%) were paired; of these:
    98583 (7.63%) aligned concordantly 0 times
    1175135 (90.93%) aligned concordantly exactly 1 time
    18625 (1.44%) aligned concordantly >1 times
    ----
    98583 pairs aligned concordantly 0 ti

CalledProcessError: Command 'b'mkdir -p /content/alignment_test/ \n\n\nfor r1 in /content/drive/MyDrive/RNAseq_Project/chrX_data/samples/*_1.fastq.gz\ndo\nr2=${r1/_1.fastq.gz/_2.fastq.gz}\n\n\nsample=$(basename "$r1" _1.fastq.gz)\n\necho "Processing $sample"\n\nhisat2 -p 2 \\\n-x "/content/drive/MyDrive/RNAseq_Project/chrX_data/indexes/chrX_tran" \\\n-1 "$r1" \\\n-2 "$r2" \\\n-S "/content/alignment_test/${sample}.sam"\n\nsamtools view -b \\\n"/content/alignment_test/${sample}.sam" \\\n>"/content/alignment_test/${sample}.bam"\n\n\nrm "/content/alignment_test/${sample}.sam"\n\n\ncp "/content/alignment_test/${sample}.bam" \\ "content/drive/MyDrive/RNAseq_Project/BAM_results/"\n\ndone\n'' returned non-zero exit status 1.

In [ ]:
!ls -lh /content/alignment_test/

total 3.1G
-rw-r--r-- 1 root root 211M Aug 22 15:04 ERR188044.bam
-rw-r--r-- 1 root root 211M Aug 22 15:58 ERR188044_chrX.bam
-rw-r--r-- 1 root root 747M Aug 22 14:46 ERR188044.sam
-rw-r--r-- 1 root root 204M Aug 22 15:59 ERR188104_chrX.bam
-rw-r--r-- 1 root root 262M Aug 22 16:01 ERR188234_chrX.bam
-rw-r--r-- 1 root root 143M Aug 22 16:02 ERR188245_chrX.bam
-rw-r--r-- 1 root root 158M Aug 22 16:03 ERR188257_chrX.bam
-rw-r--r-- 1 root root  93M Aug 22 16:04 ERR188273_chrX.bam
-rw-r--r-- 1 root root 213M Aug 22 16:05 ERR188337_chrX.bam
-rw-r--r-- 1 root root 152M Aug 22 16:06 ERR188383_chrX.bam
-rw-r--r-- 1 root root 209M Aug 22 16:08 ERR188401_chrX.bam
-rw-r--r-- 1 root root 134M Aug 22 16:09 ERR188428_chrX.bam
-rw-r--r-- 1 root root 169M Aug 22 16:10 ERR188454_chrX.bam
-rw-r--r-- 1 root root 181M Aug 22 16:11 ERR204916_chrX.bam


In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project"

total 20K
drwx------ 2 root root 4.0K Aug 22 14:53 Alignment_results
drwx------ 2 root root 4.0K Aug 22 15:08 BAM_results
drwx------ 6 root root 4.0K Aug 20 16:55 chrX_data
drwx------ 2 root root 4.0K Aug 21 16:41 FastQC_results
drwx------ 3 root root 4.0K Aug 21 17:53 MultiQC


In [ ]:
!cp /content/alignment_test/*.bam \
"/content/drive/MyDrive/RNAseq_Project/BAM_results/"

In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/BAM_results"

total 2.3G
-rw------- 1 root root 211M Aug 22 16:32 ERR188044.bam
-rw------- 1 root root 211M Aug 22 16:32 ERR188044_chrX.bam
-rw------- 1 root root 204M Aug 22 16:32 ERR188104_chrX.bam
-rw------- 1 root root 262M Aug 22 16:32 ERR188234_chrX.bam
-rw------- 1 root root 143M Aug 22 16:32 ERR188245_chrX.bam
-rw------- 1 root root 158M Aug 22 16:32 ERR188257_chrX.bam
-rw------- 1 root root  93M Aug 22 16:32 ERR188273_chrX.bam
-rw------- 1 root root 213M Aug 22 16:32 ERR188337_chrX.bam
-rw------- 1 root root 152M Aug 22 16:32 ERR188383_chrX.bam
-rw------- 1 root root 209M Aug 22 16:32 ERR188401_chrX.bam
-rw------- 1 root root 134M Aug 22 16:32 ERR188428_chrX.bam
-rw------- 1 root root 169M Aug 22 16:32 ERR188454_chrX.bam
-rw------- 1 root root 181M Aug 22 16:32 ERR204916_chrX.bam


In [ ]:
!rm "/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188044.bam"

In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/BAM_results"

total 2.1G
-rw------- 1 root root 211M Aug 22 16:32 ERR188044_chrX.bam
-rw------- 1 root root 204M Aug 22 16:32 ERR188104_chrX.bam
-rw------- 1 root root 262M Aug 22 16:32 ERR188234_chrX.bam
-rw------- 1 root root 143M Aug 22 16:32 ERR188245_chrX.bam
-rw------- 1 root root 158M Aug 22 16:32 ERR188257_chrX.bam
-rw------- 1 root root  93M Aug 22 16:32 ERR188273_chrX.bam
-rw------- 1 root root 213M Aug 22 16:32 ERR188337_chrX.bam
-rw------- 1 root root 152M Aug 22 16:32 ERR188383_chrX.bam
-rw------- 1 root root 209M Aug 22 16:32 ERR188401_chrX.bam
-rw------- 1 root root 134M Aug 22 16:32 ERR188428_chrX.bam
-rw------- 1 root root 169M Aug 22 16:32 ERR188454_chrX.bam
-rw------- 1 root root 181M Aug 22 16:32 ERR204916_chrX.bam


# BAM Sorting

The BAM files generated from HISAT2 alignments were sorted using SAMtools.

Sorting arranges aligned reads according to their genomic coordinates, which is required for downstream analysis and indexing.

Command used:

samtools sort input.bam -o output.sorted.bam

Output: 12 sorted BAM files

In [ ]:
!mkdir -p "/content/drive/MyDrive/RNAseq_Project/Sorted_BAM"

In [ ]:
%%bash

for bam in /content/drive/MyDrive/RNAseq_Project/BAM_results/*.bam
do

sample=$(basename "$bam" .bam)

echo "Sorting $sample"

samtools sort \
"$bam" \
-o "/content/drive/MyDrive/RNAseq_Project/Sorted_BAM/${sample}.sorted.bam"

done

Sorting ERR188044_chrX
Sorting ERR188104_chrX
Sorting ERR188234_chrX
Sorting ERR188245_chrX
Sorting ERR188257_chrX
Sorting ERR188273_chrX
Sorting ERR188337_chrX
Sorting ERR188383_chrX
Sorting ERR188401_chrX
Sorting ERR188428_chrX
Sorting ERR188454_chrX
Sorting ERR204916_chrX


[bam_sort_core] merging from 1 files and 1 in-memory blocks...


In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/Sorted_BAM"

total 1.5G
-rw------- 1 root root 144M Aug 22 16:45 ERR188044_chrX.sorted.bam
-rw------- 1 root root 139M Aug 22 16:45 ERR188104_chrX.sorted.bam
-rw------- 1 root root 185M Aug 22 16:45 ERR188234_chrX.sorted.bam
-rw------- 1 root root 101M Aug 22 16:46 ERR188245_chrX.sorted.bam
-rw------- 1 root root 113M Aug 22 16:46 ERR188257_chrX.sorted.bam
-rw------- 1 root root  67M Aug 22 16:46 ERR188273_chrX.sorted.bam
-rw------- 1 root root 152M Aug 22 16:46 ERR188337_chrX.sorted.bam
-rw------- 1 root root 104M Aug 22 16:46 ERR188383_chrX.sorted.bam
-rw------- 1 root root 145M Aug 22 16:46 ERR188401_chrX.sorted.bam
-rw------- 1 root root  94M Aug 22 16:47 ERR188428_chrX.sorted.bam
-rw------- 1 root root 117M Aug 22 16:47 ERR188454_chrX.sorted.bam
-rw------- 1 root root 127M Aug 22 16:47 ERR204916_chrX.sorted.bam


# BAM Indexing

After sorting the BAM files, the next step was to create index files using `samtools index`.

## Need of BAM indexing

A BAM file contains millions of aligned sequencing reads. Even though the file is sorted by genomic position, searching for a specific region directly from the BAM file would require scanning through a large amount of data.

Indexing creates a companion `.bai` file that acts like a table of contents for the BAM file. This allows bioinformatics tools to quickly locate reads aligned to a particular genomic region without reading the entire file.

## Benefits

* Enables rapid access to specific genomic coordinates.
* Improves performance of downstream analysis tools.
* Required by many visualization and analysis programs such as IGV.
* Allows efficient querying of alignment data using SAMtools.

## Command Used

```bash
samtools index sample.sorted.bam
```

## Output

For each sorted BAM file:

```text
sample.sorted.bam
sample.sorted.bam.bai
```

The `.bai` file is the index associated with the corresponding sorted BAM file.


In [ ]:
!mkdir -p "/content/drive/MyDrive/RNAseq_Project/BAM_Indexes"

In [ ]:
%%bash

for bam in /content/drive/MyDrive/RNAseq_Project/Sorted_BAM/*.sorted.bam
do

echo "Indexing $(basename "$bam")"

samtools index "$bam"

done

Indexing ERR188044_chrX.sorted.bam
Indexing ERR188104_chrX.sorted.bam
Indexing ERR188234_chrX.sorted.bam
Indexing ERR188245_chrX.sorted.bam
Indexing ERR188257_chrX.sorted.bam
Indexing ERR188273_chrX.sorted.bam
Indexing ERR188337_chrX.sorted.bam
Indexing ERR188383_chrX.sorted.bam
Indexing ERR188401_chrX.sorted.bam
Indexing ERR188428_chrX.sorted.bam
Indexing ERR188454_chrX.sorted.bam
Indexing ERR204916_chrX.sorted.bam


In [ ]:
!ls -lh "/content/drive/MyDrive/RNAseq_Project/Sorted_BAM"

total 1.5G
-rw------- 1 root root 144M Aug 22 16:45 ERR188044_chrX.sorted.bam
-rw------- 1 root root 163K Aug 22 16:54 ERR188044_chrX.sorted.bam.bai
-rw------- 1 root root 139M Aug 22 16:45 ERR188104_chrX.sorted.bam
-rw------- 1 root root 161K Aug 22 16:54 ERR188104_chrX.sorted.bam.bai
-rw------- 1 root root 185M Aug 22 16:45 ERR188234_chrX.sorted.bam
-rw------- 1 root root 167K Aug 22 16:54 ERR188234_chrX.sorted.bam.bai
-rw------- 1 root root 101M Aug 22 16:46 ERR188245_chrX.sorted.bam
-rw------- 1 root root 146K Aug 22 16:54 ERR188245_chrX.sorted.bam.bai
-rw------- 1 root root 113M Aug 22 16:46 ERR188257_chrX.sorted.bam
-rw------- 1 root root 155K Aug 22 16:54 ERR188257_chrX.sorted.bam.bai
-rw------- 1 root root  67M Aug 22 16:46 ERR188273_chrX.sorted.bam
-rw------- 1 root root 132K Aug 22 16:54 ERR188273_chrX.sorted.bam.bai
-rw------- 1 root root 152M Aug 22 16:46 ERR188337_chrX.sorted.bam
-rw------- 1 root root 166K Aug 22 16:54 ERR188337_chrX.sorted.bam.bai
-rw------- 1 root root 

In [ ]:
!find "/content/drive/MyDrive/RNAseq_Project" -type f | sort

/content/drive/MyDrive/RNAseq_Project/Alignment_results/ERR188044.sam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188044_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188104_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188234_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188245_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188257_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188273_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188337_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188383_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188401_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188428_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR188454_chrX.bam
/content/drive/MyDrive/RNAseq_Project/BAM_results/ERR204916_chrX.bam
/content/drive/MyDrive/RNAseq_Project/chrX_data/genes/chrX.gtf
/content/drive/MyDrive/RNAseq_Project/c

In [ ]:
!rm "/content/drive/MyDrive/RNAseq_Project/Alignment_results/ERR188044.sam"

In [ ]:
!rm -r "/content/drive/MyDrive/RNAseq_Project/Alignment_results"